# Axis 1 — Module correctness (`00_module_outputs`)

**What this notebook is.** The Axis-1 checks from `TESTING.md`: every module is
verified by its *output* (a deterministic artifact against a hand/known answer)
and/or its *invariants* (properties that must always hold). No chains except
where an invariant needs one.

**How to read it.** This is a *phase-1* artifact (TESTING.md §3.1): you run it
top to bottom and **read the printed comparisons with your own eyes**. Each
check prints the analytic/expected value next to the observed one and a PASS/FAIL
verdict. A green verdict is not the point — *seeing the two numbers agree* is.

**Coverage (modules that exist today):** data_prep · node_models · priors ·
constraints · mcmc_adapt · likelihood · gibbs_samplers. `diagnostics` and
`posterior` are not built yet; their cells are stubbed at the end.


## 0. Setup


In [ ]:
import sys, pathlib
!{sys.executable} -m pip install -e "C:\Users\Manonfi\OneDrive - James Madison University\Spring 26 classes & projects\Practicum Python IRtree package\PACKAGE\Irtree"

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
import irtree.tree_spec as ts
import irtree.data_prep as dp
import irtree.node_models as nm
import irtree.priors as pr
import irtree.constraints as con
import irtree.mcmc_adapt as ma
import irtree.likelihood as lk
import irtree.gibbs_samplers as gs


ROOT = pathlib.Path.cwd().parents[1]
import importlib.util
harness_path = ROOT / "tests" / "_harness.py"
spec = importlib.util.spec_from_file_location("_harness", harness_path)
h = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h)



def verdict(ok, label):
    print(f"  [{'PASS' if ok else 'FAIL'}]  {label}")
    assert ok, label


## 1. `data_prep` — output check (hand-computed `Y_star` / `T`)

**Why check directly:** this is the foundation every downstream statistic is
indexed by. A bug here is silent and poisons everything.

**The hand example.** Dichotomous disengagement tree, mapping (rows =
categories 1,2,3; cols = node 1, node 2):

```
cat 1 -> node1=1,  node2=NA     (rapid guess: path stops at node 1)
cat 2 -> node1=0,  node2=0      (engaged, incorrect)
cat 3 -> node1=0,  node2=1      (engaged, correct)
```

Take a 3-person x 2-item response matrix and write out, by hand, what the
pseudo-responses `Y_star` and the relevance masks `T` must be:


In [ ]:
spec = ts.disengagement_tree(binary=True)
print('mapping (rows=cat 1,2,3 ; cols=node1,node2):')
print(spec.mapping)

Y = np.array([[1, 2],
              [3, 1],
              [2, 3]])
print('\nY (3 persons x 2 items):'); print(Y)


### Hand-derived answer

For each cell, follow its category down the mapping:

| person,item | category | node1 (Y*) | node1 (T) | node2 (Y*) | node2 (T) |
|---|---|---|---|---|---|
| (0,0) | 1 | 1 | relevant | — | NOT relevant (path stopped) |
| (0,1) | 2 | 0 | relevant | 0 | relevant |
| (1,0) | 3 | 0 | relevant | 1 | relevant |
| (1,1) | 1 | 1 | relevant | — | NOT relevant |
| (2,0) | 2 | 0 | relevant | 0 | relevant |
| (2,1) | 3 | 0 | relevant | 1 | relevant |

So node 1 is relevant for everyone (`T1` all ones); node 2 is relevant only
where the person did **not** rapid-guess.


In [ ]:
# hand answers
Ystar1_hand = np.array([[1,0],[0,1],[0,0]], float)
T1_hand     = np.array([[1,1],[1,1],[1,1]], bool)
# node 2: NA where category==1 (rapid guess); Y* is 0 for cat 2, 1 for cat 3
Ystar2_hand = np.array([[np.nan,0],[1,np.nan],[0,1]], float)
T2_hand     = np.array([[0,1],[1,0],[1,1]], bool)

data = dp.expand_data(Y, spec, categories_start=1)

print('node 1  Y_star:'); print(data.Y_star[0])
print('node 1  T:');      print(data.T[0].astype(int))
print('node 2  Y_star:'); print(data.Y_star[1])
print('node 2  T:');      print(data.T[1].astype(int))

# compare (nan-aware for Y*)
def eq_nan(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return np.all((a == b) | (np.isnan(a) & np.isnan(b)))

verdict(eq_nan(data.Y_star[0], Ystar1_hand), 'node1 Y_star matches hand')
verdict(np.array_equal(data.T[0], T1_hand),   'node1 T matches hand')
verdict(eq_nan(data.Y_star[1], Ystar2_hand), 'node2 Y_star matches hand')
verdict(np.array_equal(data.T[1], T2_hand),   'node2 T matches hand')


### `data_prep` invariant — index sets agree with the mask

`row_idx`/`col_idx` must enumerate exactly the True cells of `T`. This is the
contract every sampler relies on.


In [ ]:
for r in range(data.n_nodes):
    ri, ci = data.row_idx[r], data.col_idx[r]
    rebuilt = np.zeros_like(data.T[r]); rebuilt[ri, ci] = True
    verdict(np.array_equal(rebuilt, data.T[r]), f'node{r} index set == mask')
    verdict(len(ri) == int(data.T[r].sum()),    f'node{r} index count == T.sum()')


## 2. `node_models` — output check (derived mixed-format structure)

**Why check directly:** the `log_prob` numbers are already exercised by the unit
conditionals (§4.1 below). What is *not* covered elsewhere is the derived
structure a mixed node computes from `pseudo_item_outcomes` — and wrong values
there crash the sampler obscurely.


In [ ]:
# items: two binary (2 outcomes), two trichotomous (3 outcomes)
node = nm.GRMNode(pseudo_item_outcomes=np.array([2, 2, 3, 3]))
print('is_mixed :', node.is_mixed,   '  (expect True)')
print('bin_idx  :', node.bin_idx,    '  (expect [0 1])')
print('poly_idx :', node.poly_idx,   '  (expect [2 3])')
verdict(node.is_mixed is True or node.is_mixed == True, 'mixed detected')
verdict(list(node.bin_idx)  == [0,1], 'bin_idx correct')
verdict(list(node.poly_idx) == [2,3], 'poly_idx correct')

# a HOMOGENEOUS node must NOT be flagged mixed
homo = nm.GRMNode(n_outcomes=3)
verdict(not homo.is_mixed, 'homogeneous node is not mixed')


### `node_models` invariant — `log_prob` is NaN only in irrelevant cells

`log_prob` is allowed to return NaN, but *only* where `T` is False. A NaN in a
relevant cell would be a real bug (and the `np.where` mask in `likelihood`
relies on this being the only place NaNs live).


In [ ]:
model = nm.ProbitRaschNode()
P, I = data.n_persons, data.n_items
lp = model.log_prob(data.Y_star[1], np.zeros(P), np.zeros(I))
rel = data.T[1]
verdict(np.all(np.isfinite(lp[rel])), 'log_prob finite in every RELEVANT cell')
print('  (NaNs in irrelevant cells are expected:', int(np.sum(~np.isfinite(lp[~rel]))), 'of', int((~rel).sum()), ')')


## 3. `priors` — output check (the *decisions*, not the numbers)

We do not check what a prior returns numerically (trivial). We check the
branch points that were bugs: the documented defaults, and the conjugacy gate.


In [ ]:
models = [nm.ProbitTwoPLNode(), nm.ProbitTwoPLNode()]
p = pr.build_default_priors(spec, models)

# documented defaults present for the parameters a 2PL tree has
for key in ['theta', 'b', 'a']:
    got = p.resolve(key, 0)
    print(f'  default[{key}] = {type(got).__name__}')
verdict(p.resolve('a', 0) is not None, 'discrimination prior assigned for 2PL')


### `priors` invariant — the conjugacy gate fires correctly

A `PriorSpec` in the variance slot of a **bounded** prior (discrimination `a`)
is not conjugate. `_check_variance_conjugacy` must **raise** by default and
**pass with a warning** under `force=True`. An unbounded-Normal variance must
pass silently.


In [ ]:
import warnings
bad = pr.TruncNormalPrior(mean=1.0, variance=pr.InvGammaPrior(1.0, 1.0), lower=0.0)

raised = False
try:
    pr._check_variance_conjugacy('a', bad, force=False)
except TypeError as e:
    raised = True
    print('  default -> TypeError (good):', str(e).splitlines()[0][:70], '...')
verdict(raised, 'bounded prior + PriorSpec variance raises without force')

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    pr._check_variance_conjugacy('a', bad, force=True)
    forced_ok = len(w) >= 1 and issubclass(w[0].category, UserWarning)
verdict(forced_ok, 'force=True downgrades to a UserWarning')

# unbounded Normal variance must NOT raise
good = pr.NormalPrior(mean=0.0, variance=pr.InvGammaPrior(1.0, 1.0))
ok = True
try:
    pr._check_variance_conjugacy('b', good, force=False)
except Exception:
    ok = False
verdict(ok, 'unbounded Normal + PriorSpec variance passes (conjugate)')


## 4. `constraints` — invariant check (validation rejects illegal models)

The grouping arithmetic surfaces in recovery, so we do not re-check it here. The
**validation** must be checked directly because a *missing* rejection is silent
— it lets an illegal model run. We confirm a covariate declared on only some
members of a sharing group is rejected (matches `_group_and_validate`).


In [ ]:
# Call the grouping validator directly on a partial declaration.
# groups_of = one shared group containing both nodes; raw = node 0 declares a
# person covariate, node 1 declares nothing. A shared coefficient cannot be
# built from a declaration present on only some members -> must be rejected.
raw       = [np.array([0, 1]), None]
groups_of = [[0, 1]]

rejected = False
try:
    dp._group_and_validate(raw, groups_of, 'person covariate')
except ValueError as e:
    rejected = True
    print('  partial declaration -> rejected (good):')
    print('    ', str(e).splitlines()[0][:75])
verdict(rejected, 'covariate on only some members of a shared group is rejected')

# and the legal case: identical declarations on both members must PASS
ok = True
try:
    dp._group_and_validate([np.array([0,1]), np.array([0,1])], groups_of, 'person covariate')
except Exception:
    ok = False
verdict(ok, 'identical declarations across the group are accepted')


## 5. `mcmc_adapt` — output check (step-size moves the right way, freezes after warmup)

Robbins-Monro: too many accepts should **grow** the step (propose bigger moves);
too many rejects should **shrink** it. After warmup the step must **freeze**.


In [ ]:
n_warmup = 200
st = ma.make_adapt_state(n_warmup)
step0 = st.step_size

# feed 100 ACCEPTS during warmup -> step should grow
s = st
for t in range(100):
    s = ma.adapt_step(s, True, t)
grew = s.step_size > step0
print(f'  100 accepts: step {step0:.4f} -> {s.step_size:.4f}  (expect grow)')
verdict(grew, 'accepts grow the step size')

# fresh state, feed 100 REJECTS -> step should shrink
s = ma.make_adapt_state(n_warmup)
for t in range(100):
    s = ma.adapt_step(s, False, t)
shrank = s.step_size < step0
print(f'  100 rejects: step {step0:.4f} -> {s.step_size:.4f}  (expect shrink)')
verdict(shrank, 'rejects shrink the step size')

# after warmup the step is frozen: an accept past n_warmup must not change it
s = ma.make_adapt_state(n_warmup)
for t in range(n_warmup):
    s = ma.adapt_step(s, True, t)
frozen_before = s.step_size
s_after = ma.adapt_step(s, True, n_warmup + 5)
verdict(abs(s_after.step_size - frozen_before) < 1e-12, 'step frozen after warmup')


## 6. `likelihood` — invariant check (cache total == sum; recompute locality; NaN-safe)

`likelihood` has no standalone output to hand-verify — it has invariants:
the cached total always equals the sum of node scalars, a recompute touches
only its target node, and the whole thing stays finite despite NaN in
irrelevant cells (the `np.where` mask, not `T*logprob`).


In [ ]:
R = data.n_nodes
theta = [np.zeros(P) for _ in range(R)]
b = [np.zeros(I) for _ in range(R)]
models = [nm.ProbitRaschNode(), nm.ProbitRaschNode()]
cache = lk.init_cache(data, theta, b, models)

# finiteness (the 0.0*nan bug made this NaN)
verdict(np.all(np.isfinite(cache.node_logliks)), 'all node log-liks finite (np.where mask)')

# total == sum
verdict(abs(cache.total - float(np.sum(cache.node_logliks))) < 1e-9, 'cache.total == sum(node_logliks)')

# perturb one entry; total still tracks
cache.node_logliks[0] += 3.5
verdict(abs(cache.total - float(np.sum(cache.node_logliks))) < 1e-9, 'total tracks after perturbation')

# recompute locality: recompute node 1, node 0 must be untouched
cache = lk.init_cache(data, theta, b, models)
before = cache.node_logliks.copy()
lk.recompute_node(cache, 1, data, np.full(P, 0.7), b[1], models[1])
verdict(cache.node_logliks[0] == before[0], 'recompute(1) leaves node 0 untouched')
verdict(cache.node_logliks[1] != before[1], 'recompute(1) updates node 1')


## 7. `gibbs_samplers` — output check (the unit conditionals)

A Gibbs draw *is* this module's output; its known answer is the analytic full
conditional, re-derived **from the model** in `_harness` (never from the code).
We freeze all conditioning variables, draw many times, and print the sampled
mean/variance next to the analytic mean/variance. Agreement within a few
Monte-Carlo standard errors is the proof the sufficient statistics were written
correctly.

This section reproduces the core of `test_unit_conditionals.py` in an
*inspectable* form — you see the two numbers, not just a checkmark. Set
`N_DRAWS` small for a quick pass, large for a strict one.


In [ ]:
N_DRAWS = 20000     # bump to 60000 for a strict read

def show_moment(draws, mu, var, label):
    draws = np.asarray(draws)
    emp_m = draws.mean(0); emp_v = draws.var(0, ddof=1)
    se = np.sqrt(draws.var(0, ddof=1) / draws.shape[0])
    zmax = np.max(np.abs(emp_m - mu) / np.where(se==0,1e-12,se))
    print(f'  {label}')
    print(f'    analytic mean : {np.round(np.atleast_1d(mu),4)}')
    print(f'    sampled  mean : {np.round(np.atleast_1d(emp_m),4)}   (max {zmax:.2f} SE off)')
    print(f'    analytic var  : {np.round(np.atleast_1d(var),4)}')
    print(f'    sampled  var  : {np.round(np.atleast_1d(emp_v),4)}')
    verdict(zmax < 4.0, label + ' mean within 4 SE')


### 7.1 `gibbs_b` (1PL) — precision `1/τ² + n`, statistic `Σ(θ − Z)`


In [ ]:
rng = np.random.default_rng(1)
Pp, Ii = 120, 6
theta_f = rng.normal(0,1,Pp); b_f = rng.normal(0,1,Ii)
T = rng.random((Pp,Ii)) > 0.25; ri, ci = np.where(T)
Z = rng.normal(h.simulate_probit_cell_means(theta_f, b_f), 1.0)
tau2 = 4.0

draws = h.draw_many(lambda r: gs.gibbs_b(Z, theta_f, tau2, T, ri, ci, r), N_DRAWS)
mu = np.empty(Ii); var = np.empty(Ii)
for i in range(Ii):
    mu[i], var[i] = h.analytic_b_scalar(Z[:,i], theta_f, T[:,i], tau2)
show_moment(draws, mu, var, 'gibbs_b (1PL)')


### 7.2 `gibbs_b` (2PL) — precision now carries `a²`, statistic carries `a`


In [ ]:
rng = np.random.default_rng(2)
a_f = rng.uniform(0.6, 1.7, Ii)
Z2 = rng.normal(h.simulate_probit_cell_means(theta_f, b_f, a=a_f), 1.0)
draws = h.draw_many(lambda r: gs.gibbs_b(Z2, theta_f, tau2, T, ri, ci, r, a=a_f), N_DRAWS)
mu = np.empty(Ii); var = np.empty(Ii)
for i in range(Ii):
    mu[i], var[i] = h.analytic_b_scalar(Z2[:,i], theta_f, T[:,i], tau2, a_i=a_f[i])
show_moment(draws, mu, var, 'gibbs_b (2PL, a-weighted)')

# and the a=None regression: must match the 1PL call bitwise
r1 = gs.gibbs_b(Z, theta_f, tau2, T, ri, ci, np.random.default_rng(9))
r2 = gs.gibbs_b(Z, theta_f, tau2, T, ri, ci, np.random.default_rng(9), a=None)
verdict(np.array_equal(r1, r2), 'a=None is bitwise identical to 1PL')


### 7.3 θ via the cross-node path (there is no per-node `gibbs_theta`)

`compute_suff_stats` → `gibbs_theta_correlated` with a diagonal Σ must equal the
scalar posterior per person.


In [ ]:
rng = np.random.default_rng(4)
sigma2 = 1.3
Sigma = gs.build_sigma(np.array([sigma2]), np.eye(1))
def draw_theta(r):
    S, n = gs.compute_suff_stats([Z],[b_f],[T],[ri],[ci],[False])
    S, n = gs.reduce_suff_stats_to_dims(S, n, [[0]])
    return gs.gibbs_theta_correlated(S, n, Sigma, r)[:,0]
draws = h.draw_many(draw_theta, N_DRAWS)
mu = np.empty(Pp); var = np.empty(Pp)
for p in range(Pp):
    mu[p], var[p] = h.analytic_theta_scalar(Z[p], b_f, T[p], sigma2)
# summarize across persons (too many to print each)
emp_m = draws.mean(0); se = np.sqrt(draws.var(0,ddof=1)/N_DRAWS)
zmax = np.max(np.abs(emp_m - mu)/np.where(se==0,1e-12,se))
print(f'  theta 1PL: max {zmax:.2f} SE off across {Pp} persons')
print(f'    example person 0: analytic {mu[0]:.4f} / sampled {emp_m[0]:.4f}')
verdict(zmax < 4.5, 'theta 1PL mean within tolerance (all persons)')


### 7.4 A covariate coefficient — `gibbs_coef_pooled` vs the brute-force regression

The cleanest known answer for a coefficient is the full-design Normal-Normal
posterior: stack one design row per relevant cell, apply the standard formula.


In [ ]:
rng = np.random.default_rng(41)
Pp2, Ii2, K = 200, 12, 2
theta_c = rng.normal(0,1,Pp2); b_c = rng.normal(0,1,Ii2)
X = rng.normal(size=(Pp2,K)); beta = rng.normal(size=K)
Tc = rng.random((Pp2,Ii2)) > 0.25; ric, cic = np.where(Tc)
means = theta_c[ric] - b_c[cic] + (X @ beta)[ric]
Zc = np.zeros((Pp2,Ii2)); Zc[ric,cic] = rng.normal(means, 1.0)
tau2b = 3.0; Xobs = np.ones(Pp2, bool)

# run_chain draws a coefficient from each node's latent residual with
# gibbs_coef_pooled(kind, nodes, residuals, masks, discriminations, design, tau2, rng);
# kind 'x' = person covariates (beta), 'q' = item covariates (kappa).
E = np.zeros((Pp2,Ii2)); E[ric,cic] = Zc[ric,cic] - (theta_c[ric] - b_c[cic])   # latent residual
draws = h.draw_many(lambda r: gs.gibbs_coef_pooled('x', [0], [E], [Tc], [None], X, tau2b, r), N_DRAWS)
y = Zc[ric,cic] - (theta_c[ric] - b_c[cic])   # residual
design = X[ric]
mu, V = h.analytic_normal_normal(design, y, tau2b)
emp_m = draws.mean(0); emp_cov = np.cov(draws.T)
print('  analytic beta mean :', np.round(mu,4))
print('  sampled  beta mean :', np.round(emp_m,4))
print('  analytic beta cov  :\n', np.round(V,5))
print('  sampled  beta cov  :\n', np.round(emp_cov,5))
se = np.sqrt(np.diag(emp_cov)/N_DRAWS)
verdict(np.max(np.abs(emp_m-mu)/se) < 4.0, 'gibbs_coef_pooled (beta) mean matches brute-force design')


## 8. `diagnostics` — output check (against constructed chains with known answers)

These are tested on chains we *build* with a known R-hat / ESS, so the check is
separated from any real fit. Convention: input is `[S, C]` = samples × chains.


In [ ]:
import irtree.diagnostics as dg

def ar1(n, rho, seed):
    r = np.random.default_rng(seed); x = np.zeros(n)
    sd = np.sqrt(1 - rho**2)
    for t in range(1, n): x[t] = rho*x[t-1] + r.normal(0, sd)
    return x

N = 4000
# identical chains -> R-hat ~ 1 ; divergent means -> R-hat >> 1
ident = np.column_stack([np.random.default_rng(1).normal(size=N)]*4)
div   = np.column_stack([np.random.default_rng(s).normal(4*s, 1, N) for s in range(4)])
print(f'  R-hat identical chains : {dg.rhat(ident):.4f}   (expect ~1)')
print(f'  R-hat divergent chains : {dg.rhat(div):.3f}    (expect >> 1)')
verdict(abs(dg.rhat(ident)-1) < 0.02, 'R-hat ~1 on identical chains')
verdict(dg.rhat(div) > 1.5,           'R-hat large on divergent chains')


In [ ]:
# ESS on a KNOWN autocorrelation: AR(1) rho=0.5 -> ESS/N ~ (1-rho)/(1+rho) = 1/3
white = np.column_stack([np.random.default_rng(s).normal(size=N) for s in range(4)])
ar    = np.column_stack([ar1(N, 0.5, s) for s in range(4)])
ess_w,_ = dg.ess(white); ess_a,_ = dg.ess(ar)
print(f'  white-noise ESS ratio  : {ess_w/(4*N):.3f}   (expect ~1)')
print(f'  AR(1) rho=.5 ESS ratio : {ess_a/(4*N):.3f}   (expect ~0.333)')
verdict(0.8 < ess_w/(4*N) < 1.15,       'ESS ~ N for white noise')
verdict(abs(ess_a/(4*N) - 1/3) < 0.06,  'ESS matches AR(1) theory')

# lag-1 ACF recovers rho
a = dg.acf(ar1(20000, 0.6, 0), lags=[1])
print(f'  lag-1 ACF of AR(1) rho=.6 : {a[1]:.3f}')
verdict(abs(a[1]-0.6) < 0.05, 'ACF lag-1 recovers rho')


## 9. `posterior` — output check (summaries against a chain with known moments)

Feed a chain of `N(2, 1.5²)` draws; the summary must recover mean≈2, sd≈1.5, and
the 2.5/97.5 quantiles at `2 ∓ 1.96·1.5`. No model fit — a constructed chain with
a known target.


In [ ]:
import irtree.posterior as po, irtree.mcmc_sampler as ms

def fake_result(chain2d, names):
    S = chain2d.shape[0]
    return ms.MCMCResult(chain=chain2d.astype(np.float32), param_names=list(names),
        log_lik_trace=np.zeros(S), accept_rates={}, n_warmup=0,
        n_iter=S, final_params={}, rng_state={})

x = np.random.default_rng(0).normal(2.0, 1.5, size=(60000, 1))
s = po.posterior_summary(fake_result(x, ['x']))['x']
print(f'  mean  : {s.mean:.3f}  (expect 2.000)')
print(f'  sd    : {s.sd:.3f}  (expect 1.500)')
print(f'  2.5%  : {s.q025:.3f}  (expect {2-1.96*1.5:.3f})')
print(f'  97.5% : {s.q975:.3f}  (expect {2+1.96*1.5:.3f})')
verdict(abs(s.mean-2)<0.03 and abs(s.sd-1.5)<0.03, 'summary recovers known mean/sd')
verdict(abs(s.q025-(2-1.96*1.5))<0.06 and abs(s.q975-(2+1.96*1.5))<0.06, 'quantiles match N(2,1.5)')


## 10. `fit` — invariant check (orchestration: runs, structured, reproducible)

`irtree_fit` is the orchestration wrapper — nothing to hand-verify in isolation,
so it is checked by invariants through a tiny real fit: it runs, returns the
expected surface, stays finite, and is reproducible under a fixed seed. (A
*recovery* claim on its estimates is Axis 2, not here.)


In [ ]:
import warnings, irtree.fit as ft
from scipy.special import ndtr

rng = np.random.default_rng(0); P, I = 120, 8
th1=rng.normal(0,1,P); th2=rng.normal(0,1,P); b1=rng.normal(0,.7,I); b2=rng.normal(0,1,I)
rg=rng.random((P,I))<ndtr(th1[:,None]-b1[None,:]); co=rng.random((P,I))<ndtr(th2[:,None]-b2[None,:])
Yf=np.where(rg,1,np.where(co,3,2))
specf=ts.disengagement_tree(binary=True); dataf=dp.expand_data(Yf,specf,categories_start=1)
modsf=[nm.ProbitRaschNode(), nm.ProbitRaschNode()]

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    # Short diagnostic chains need warmup_mode='fixed': the default evidence mode
    # requires n_iter >= 1600 (600 warm-up + 1000 kept draws).
    f1 = ft.irtree_fit(dataf, specf, modsf, n_iter=80, n_warmup=20, warmup_mode='fixed', n_chains=2, seed=7)
    f2 = ft.irtree_fit(dataf, specf, modsf, n_iter=80, n_warmup=20, warmup_mode='fixed', n_chains=2, seed=7)

verdict(len(f1.results) == 2, 'stores the requested number of chains')
verdict(all(np.all(np.isfinite(np.asarray(r.chain,float))) for r in f1.results), 'chains finite')
same = all(np.array_equal(np.asarray(a.chain), np.asarray(b.chain)) for a,b in zip(f1.results,f2.results))
verdict(same, 'same seed -> identical chains (reproducible)')

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    f3 = ft.irtree_fit(dataf, specf, modsf, n_iter=80, n_warmup=20, warmup_mode='fixed', n_chains=2, seed=99)
diff = not all(np.array_equal(np.asarray(a.chain), np.asarray(b.chain)) for a,b in zip(f1.results,f3.results))
verdict(diff, 'different seed -> different chains (RNG threaded)')

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    w = f1.waic()
wn = [v for v in vars(w).values() if isinstance(v,(int,float))]
verdict(len(wn)>0 and all(np.isfinite(v) for v in wn), 'WAIC fields finite')


## Summary

If every cell above printed **PASS**, Axis 1 is green: each module's output
matches its hand/known answer and every invariant holds. That means when an
Axis-2 recovery fails, the setup (data prep, priors, constraints, grouping,
the individual draws) is already known-good — so the failure is an *estimation*
problem, localized by elimination.

Phase 2: once you have watched this run and trust it, the same checks live
headless in `tests/test_module_outputs.py`, `test_unit_conditionals.py`, and
`test_invariants.py`, importing the same `_harness`.
